In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import scipy.sparse as sp
import anndata as ad
import gseapy as gp
from copy import deepcopy
import scanpy.external as sce
import pickle

# sc.logging.print_version()
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5,5))

import chronopack as cp


In [2]:
adata_hvg = sc.read_h5ad('../output/adata_hvg.chk1.h5ad')
adata = sc.read_h5ad('../output/adata_full.chk1.h5ad')

# Compute CPM & logCPM matrices for negative bulk control

In [ ]:
NEGATIVE_CONTROL_PATH = '../intmd/k562_neg-ctrl_mtx-agg.pkl'


# Compute pseudobulk CPM & logCPM matrices for LPS+THP1 experiment

In [ ]:
# Tag-bias candidate identification below computes temporal pseudobulk internally.


# Prioritize genes by variation over time

In [ ]:
# Trajectory clusters 0 and 1 are K562 states; cluster 2 is THP1.
cell_type_by_traj_cluster = {'0': 'K562', '1': 'K562', '2': 'THP1'}
adata.obs['cell_type'] = adata.obs['traj_clusters'].apply(cell_type_by_traj_cluster.get)


# New approach

In [6]:
adata_thp1_hvg = adata[(adata.obs['cell_type'] == 'THP1'), :].copy()
adata_k562_hvg = adata[(adata.obs['cell_type'] == 'K562'), :].copy()

In [ ]:
# Per-cell-type temporal pseudobulk is computed inside cp.tl.identify_tag_bias_candidates().


In [8]:
adata_k562_hvg.var["total_counts"] = np.asarray(
    adata_k562_hvg.layers["counts"].sum(axis=0)
).ravel()

In [9]:
adata_thp1_hvg.var["total_counts"] = np.asarray(
    adata_thp1_hvg.layers["counts"].sum(axis=0)
).ravel()

In [10]:
low_count_genes_k562 = set(adata_k562_hvg.var.index[adata_k562_hvg.var["total_counts"] < 10])
low_count_genes_thp1 = set(adata_thp1_hvg.var.index[adata_thp1_hvg.var["total_counts"] < 10])
print(f"K562 low-count genes to remove before tradeSeq input writing: {len(low_count_genes_k562)}")
print(f"THP1 low-count genes to remove before tradeSeq input writing: {len(low_count_genes_thp1)}")


K562 low-count genes to remove before tradeSeq input writing: 7253
THP1 low-count genes to remove before tradeSeq input writing: 10695


In [ ]:
adata_hvg = adata.copy()
adata_hvg.shape


# Compute correlations

In [ ]:
corr_k562 = cp.tl.identify_tag_bias_candidates(
    adata_k562_hvg,
    NEGATIVE_CONTROL_PATH,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.8,
    local_fraction_threshold=0.5,
)

corr_thp1 = cp.tl.identify_tag_bias_candidates(
    adata_thp1_hvg,
    NEGATIVE_CONTROL_PATH,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.7,
    local_fraction_threshold=0.3,
)


In [ ]:
corr_k562 = cp.tl.filter_tag_bias_candidates(
    corr_k562,
    local_r_threshold=0.7,
    local_fraction_threshold=0.25,
    use_residual_only=True,
)

print("Tag-bias candidates:", int(corr_k562["tag_bias_candidate"].sum()))

corr_k562[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)


In [ ]:
corr_thp1 = cp.tl.filter_tag_bias_candidates(
    corr_thp1,
    local_r_threshold=0.7,
    local_fraction_threshold=0.25,
    use_residual_only=True,
)

print("Tag-bias candidates:", int(corr_thp1["tag_bias_candidate"].sum()))

corr_thp1[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)


In [ ]:
excl_genes_k562 = cp.tl.tag_bias_exclusion_genes(corr_k562)
print(len(excl_genes_k562))
print('HNRNPA1P27' in excl_genes_k562)

In [ ]:
excl_genes_thp1 = cp.tl.tag_bias_exclusion_genes(corr_thp1)
print(len(excl_genes_thp1))
print('HNRNPA1P27' in excl_genes_thp1)

# Pre-filter @K562

In [ ]:
adata_k562_hvg = cp.io.filter_technical_genes(adata_k562_hvg, exclude_genes=excl_genes_k562)
adata_k562_hvg = cp.io.filter_low_count_genes(adata_k562_hvg, min_total_counts=10, counts_layer="counts")
adata_k562_hvg


In [18]:
print('PLAU' in set(adata_k562_hvg.var.index)) # similar shape
print('CD83' in set(adata_k562_hvg.var.index)) # similar shape
print('EGR1' in set(adata_k562_hvg.var.index)) # similar shape
print('IER3' in set(adata_k562_hvg.var.index)) # similar shape

True
True
True
True


In [19]:
# TPs
print('REL' in set(adata_k562_hvg.var.index)) # real, should be included
print('ALAS1' in set(adata_k562_hvg.var.index)) # also real, should be included
print('SLC40A1' in set(adata_k562_hvg.var.index)) # similar shape but a lot noisier
print('TRAF3' in set(adata_k562_hvg.var.index)) # similar shape

# TNs
print('FTH1' in set(adata_k562_hvg.var.index)) # should have high correlation
print('RELA' in set(adata_k562_hvg.var.index)) # similar shape

True
True
True
True
True
True


# Pre-filter @THP1

In [ ]:
adata_thp1_hvg = cp.io.filter_technical_genes(adata_thp1_hvg, exclude_genes=excl_genes_thp1)
adata_thp1_hvg = cp.io.filter_low_count_genes(adata_thp1_hvg, min_total_counts=10, counts_layer="counts")
adata_thp1_hvg


In [21]:
print('PLAU' in set(adata_thp1_hvg.var.index)) # similar shape
print('CD83' in set(adata_thp1_hvg.var.index)) # similar shape
print('EGR1' in set(adata_thp1_hvg.var.index)) # similar shape
print('IER3' in set(adata_thp1_hvg.var.index)) # similar shape

True
True
True
True


In [22]:
# TPs
print('REL' in set(adata_thp1_hvg.var.index)) # real, should be included
print('ALAS1' in set(adata_thp1_hvg.var.index)) # also real, should be included
print('SLC40A1' in set(adata_thp1_hvg.var.index)) # similar shape but a lot noisier
print('TRAF3' in set(adata_thp1_hvg.var.index)) # similar shape

# TNs
print('FTH1' in set(adata_thp1_hvg.var.index)) # should have high correlation
print('RELA' in set(adata_thp1_hvg.var.index)) # similar shape

True
True
True
True
True
True


In [23]:
print('HNRNPA1P27' in adata_k562_hvg.var.index)
print('HNRNPA1P27' in adata_thp1_hvg.var.index)

False
False


# Prepare input files for TradeSeq (per cell type)

In [24]:
adata_full = sc.read_h5ad('../output/adata_full.chk1.h5ad')
adata_k562_full = adata_full[adata_k562_hvg.obs.index, :]
print(adata_k562_full.shape)

adata_thp1_full = adata_full[adata_thp1_hvg.obs.index, :]
print(adata_thp1_full.shape)

(3128, 21693)
(787, 21693)


# Prepare input files for TradeSeq (across cell types)

In [25]:
adata_k562_hvg.write_h5ad('../output/adata_k562_prefiltered.fixed.h5ad')
adata_thp1_hvg.write_h5ad('../output/adata_thp1_prefiltered.fixed.h5ad')


In [26]:
cp.tl.fit_gene_dynamics(
    adata_k562_hvg,
    output_dir='../intmd/sc-K562_to_tradeSeq.fixed',
    full_adata=adata_k562_full,
    prepare_inputs=True,
    run_backend=False,
)
cp.tl.fit_gene_dynamics(
    adata_thp1_hvg,
    output_dir='../intmd/sc-THP1_to_tradeSeq.fixed',
    full_adata=adata_thp1_full,
    prepare_inputs=True,
    run_backend=False,
)


GeneDynamicsResult(backend='tradeseq', key_added='gene_dynamics', metadata={'time_key': 'time', 'layer': 'counts', 'input_dir': '../intmd/sc-THP1_to_tradeSeq.fixed', 'n_points': 9, 'workers': 4, 'screen_nknots': 4, 'final_nknots': 8, 'p_threshold': 0.05, 'outputs': {'association': 'asso_df.1trajectory.nknots4.csv', 'significant': 'sig_df.1trajectory.nknots4.csv', 'sce': 'sce.1trajectory.nknots8.rds', 'gamList': 'gamList.1trajectory.nknots8.rds'}, 'legacy_outputs': {'association': [], 'significant': [], 'sce': [], 'gamList': []}, 'prepared_paths': {'counts': '../intmd/sc-THP1_to_tradeSeq.fixed/counts_genes_x_cells.csv.gz', 'cell_times': '../intmd/sc-THP1_to_tradeSeq.fixed/cell_times.csv.gz', 'subset_genes': '../intmd/sc-THP1_to_tradeSeq.fixed/subset_genes.csv', 'full_counts': '../intmd/sc-THP1_to_tradeSeq.fixed/counts_genes_x_cells.full.csv.gz', 'full_cell_times': '../intmd/sc-THP1_to_tradeSeq.fixed/cell_times.full.csv.gz', 'full_subset_genes': '../intmd/sc-THP1_to_tradeSeq.fixed/subset

In [27]:
# Full-library tradeSeq inputs are written by the package-backed calls above.
